In [ ]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [ ]:
%pip install torch torch_geometric
dbutils.library.restartPython()

In [ ]:
from datetime import date, timedelta

import pyspark.sql.functions as F
from pyspark.sql import Column
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.window import Window

def get_customer(spark: SparkSession, customer_shortname: str) -> DataFrame:
    """Customers/operators/publishers are identified with different ids accross the tables."""
    customers = (
        spark.table("mirakl_ai.ds_etl_prod.t2s_gold_customer")
        .where(F.col("shortName") == customer_shortname)
        .select("shortName", "publicId", F.col("databaseName").alias("db_name"))
    )

    publishers = (
        spark.table("mirakl_data_platform.prod_data_platform_silver.artemis_insights_customer").alias("I")
        .join(
            spark.table("mirakl_data_platform.prod_data_platform_silver.artemis_publisher").alias("P"),
            F.col("P.insights_customer_id") == F.col("I.id"),
            "left",
        )
        .where(F.col("I.__k8s_namespace") == "artemis-prod")
        .select(F.col("P.id").alias("publisherId"), F.col("I.public_id").alias("public_id"))
    )

    return (
        customers.join(publishers, customers.publicId == publishers.public_id, "inner")
        .select(
            F.col("shortName").alias("customer_shortname"),
            F.col("publicId").alias("customerId"),
            "db_name",
            "publisherId"
        )
    )

def get_products(spark: SparkSession, db_names: list[str]) -> DataFrame:
    """Loads the catalog of active master products (parent id = -1). To be used to get more features."""
    return (
        spark.table("mirakl_ai.ds_etl_prod.t2s_mongo_product_0_current")
        .filter(
            F.col("db_name").isin(db_names)
            & F.col("isActive")
            & F.col("isOkInStream")
            & (F.col("parentId") == -1)
        )
        .select(
            F.col("internalId").cast("bigint").alias("internalId"),
            "fwProductId",
            "name",
            "imageUrl"
        )
    )

def get_embeddings(spark: SparkSession, db_names: list[str]) -> DataFrame:
    """Loads embeddings for active master products"""
    return (
        spark.table("mirakl_ai.ds_artemis_prod.product_embeddings")
        .filter(
            F.col("db_name").isin(db_names)
            & (F.col("last_model_id") == "06bbe5791147457aa395960da021c05c")
        )
        .select(
            "internalId",
            "embeddings",
            "last_update_date"
        )
    )

def get_views(spark: SparkSession, customer_ids: list[str], start_date: date, end_date: date) -> DataFrame:
    """Loads raw tracking. Not cleaned. Merged user is used to reconcilate users ids."""
    df_merged_user = (
        spark.table("mirakl_data_platform.prod_data_platform_silver.t2s_merged_user")
        .filter(F.col("customerId").isin(customer_ids))
        .select(
            F.col("masterId").alias("userMasterId"),
            F.col("sourceId").alias("userId"),
        )
    )

    return (
        spark.table("mirakl_data_platform.prod_data_platform_silver.t2s_tracking_product_display_page_event_fct")
        .filter(
            F.col("customerId").isin(customer_ids)
            & F.col("emitted").between(start_date, end_date)
            & F.col("user.internalId").isNotNull()
            & F.col("masterProductInternalId").isNotNull()
        )
        .withColumnRenamed("masterProductInternalId", "internalId")
        .withColumn("userId", F.col("user.internalId"))
        .join(df_merged_user, on="userId", how="left")
        .withColumn("userId", F.coalesce(F.col("userMasterId"), F.col("userId")))
        .select(
            "emitted", # timestamp of the event
            "internalId", # product internalId
            "userId", # only used for aggregation, doesn't carry meaning
        )
    )

def sessionize_views(df_views: DataFrame, session_timeout: int = 1800) -> DataFrame:
    """Split user views into sessions based on an inactivity timeout. Returns views enriched with a session_id and session_start timestamp."""
    user_window = Window.partitionBy("userId").orderBy("emitted", "internalId")

    df = df_views.withColumn(
        "seconds_since_last", 
        (F.col("emitted") - F.lag("emitted").over(user_window)).cast("long")
    )

    df = df.withColumn(
        "is_new_session",
        F.when(
            F.col("seconds_since_last").isNull() | (F.col("seconds_since_last") > session_timeout),
            F.lit(1)
        ).otherwise(F.lit(0))
    )

    df = df.withColumn(
        "session_id",
        F.sum("is_new_session").over(user_window.rowsBetween(Window.unboundedPreceding, Window.currentRow))
    )

    session_window = Window.partitionBy("userId", "session_id")
    df = df.withColumn(
        "session_start",
        F.min("emitted").over(session_window)
    )

    return df.select("userId", "session_id", "internalId", "emitted", "session_start")

def split_sessions(df_sessionized: DataFrame, cutoff_train: date, cutoff_val: date) -> DataFrame:
    """Split sessions into train/val/test based on hard date cutoffs.

    session_start <  cutoff_train  -> train
    cutoff_train <= session_start <  cutoff_val  -> val
    cutoff_val   <= session_start                -> test
    """
    df_sessions = df_sessionized.select("userId", "session_id", "session_start").distinct()

    df_sessions = df_sessions.withColumn(
        "split",
        F.when(F.col("session_start") < F.lit(cutoff_train), F.lit("train"))
        .when(F.col("session_start") < F.lit(cutoff_val), F.lit("val"))
        .otherwise(F.lit("test"))
    )

    return df_sessionized.join(
        df_sessions.select("userId", "session_id", "split"),
        on=["userId", "session_id"],
        how="inner",
    )

USE_SESSION_WEIGHT_NORM = False

def build_co_view_edges(df_sessionized: DataFrame) -> DataFrame:
    """Generate co-view edges from sessionized views. Returns pairs of products seen in the same session with their co-occurrence count."""
    df_dedup = df_sessionized.select("userId", "session_id", "internalId").distinct()

    session_sizes = (
        df_dedup
        .groupBy("userId", "session_id")
        .agg(F.countDistinct("internalId").alias("session_size"))
    )

    df_dedup = df_dedup.join(session_sizes, on=["userId", "session_id"], how="inner")

    df_pairs = (
        df_dedup.alias("L")
        .join(
            df_dedup.alias("R"),
            on=(
                (F.col("L.userId") == F.col("R.userId"))
                & (F.col("L.session_id") == F.col("R.session_id"))
                & (F.col("L.internalId") < F.col("R.internalId"))
            ),
            how="inner"
        )
        .select(
            F.col("L.internalId").alias("product_A"),
            F.col("R.internalId").alias("product_B"),
            (
                (F.lit(1.0) / (F.col("L.session_size") - F.lit(1)))
                if USE_SESSION_WEIGHT_NORM
                else F.lit(1.0)
            ).alias("pair_weight"),
        )
    )

    return df_pairs.groupBy("product_A", "product_B").agg(
        F.sum("pair_weight").alias("weight"),
        F.count("*").alias("raw_count"),
    )

def remove_train_edges(df_edges: DataFrame, df_train_edges: DataFrame) -> DataFrame:
    """Remove from df_edges any pair already present in df_train_edges."""
    return df_edges.join(
        df_train_edges.select("product_A", "product_B"),
        on=["product_A", "product_B"],
        how="left_anti"
    )

In [ ]:
df_customer = get_customer(spark, customer_shortname="maisons-du-monde")
customer_rows = df_customer.collect()
customer_ids = [row["customerId"] for row in customer_rows]
publisher_ids = [row["publisherId"] for row in customer_rows]
db_names = [row["db_name"] for row in customer_rows]

df_products = get_products(spark, db_names)
df_embeddings = get_embeddings(spark, db_names)

start_date = date(2026, 1, 1)
end_date = date(2026, 5, 1)
cutoff_val = end_date - timedelta(days=15)
cutoff_train = end_date - timedelta(days=30)
assert start_date < cutoff_train < cutoff_val < end_date

df_views = get_views(spark, customer_ids, start_date, end_date)

In [ ]:
df_sessionized = sessionize_views(df_views.join(df_products, on="internalId", how="inner"))

df_split = split_sessions(df_sessionized, cutoff_train=cutoff_train, cutoff_val=cutoff_val)
df_split = df_split.cache()

def build(df_subset, df_train_edges=None):
    edges = build_co_view_edges(df_subset)
    if df_train_edges is not None:
        edges = remove_train_edges(edges, df_train_edges)
    return edges

df_train_edges = build(df_split.filter(F.col("split") == "train"))
df_val_edges = build(df_split.filter(F.col("split") == "val"), df_train_edges)
df_test_edges = build(df_split.filter(F.col("split") == "test"), df_train_edges)

def count_nodes(df_edges: DataFrame) -> int:
    return (
        df_edges.select(F.col("product_A").alias("node"))
        .union(df_edges.select(F.col("product_B").alias("node")))
        .distinct()
        .count()
    )

all_edges = df_train_edges.union(df_val_edges).union(df_test_edges)
total_nodes = count_nodes(all_edges)

print(f"Total nodes: {total_nodes}")
print(f"Train  — edges: {df_train_edges.count()}")
print(f"Val    — edges: {df_val_edges.count()}")
print(f"Test   — edges: {df_test_edges.count()}")

In [ ]:
import pandas

df_graph_nodes = (
    df_train_edges.select(F.col("product_A").alias("internalId"))
    .union(df_train_edges.select(F.col("product_B").alias("internalId")))
    .union(df_val_edges.select(F.col("product_A").alias("internalId")))
    .union(df_val_edges.select(F.col("product_B").alias("internalId")))
    .union(df_test_edges.select(F.col("product_A").alias("internalId")))
    .union(df_test_edges.select(F.col("product_B").alias("internalId")))
    .distinct()
)

df_train_edges_pandas = df_train_edges.toPandas()
df_val_edges_pandas = df_val_edges.toPandas()
df_test_edges_pandas = df_test_edges.toPandas()

emb_pandas = df_embeddings.join(df_graph_nodes, on="internalId", how="inner").toPandas()

In [ ]:
import pandas as pd
import numpy as np

def build_node_mapping(*edge_dfs: pd.DataFrame) -> dict[int, int]:
    """
    Build a mapping from internalId to contiguous index (0..N-1).
    Takes any number of edge DataFrames (train, val, test) to ensure
    all nodes are included.
    """

    all_nodes = pd.concat(
        [df["product_A"] for df in edge_dfs] + [df["product_B"] for df in edge_dfs]
    ).unique()

    all_nodes.sort()
    node2idx = {node_id: idx for idx, node_id in enumerate(all_nodes)} 

    return node2idx

def remap_edges(df: pd.DataFrame, node2idx: dict[int, int]) -> pd.DataFrame:
    """
    Replace internalIds with contiguous indices in an edge DataFrame.
    """
    df = df.copy()
    df["product_A"] = df["product_A"].map(node2idx)
    df["product_B"] = df["product_B"].map(node2idx)
    return df

In [ ]:
# Build node mapping across all 3 splits
node2idx = build_node_mapping(df_train_edges_pandas, df_val_edges_pandas, df_test_edges_pandas)

print(f"Number of nodes: {len(node2idx)}")

# Apply mapping to remap internalIds to contiguous indices
train_remapped = remap_edges(df_train_edges_pandas, node2idx)
val_remapped = remap_edges(df_val_edges_pandas, node2idx)
test_remapped = remap_edges(df_test_edges_pandas, node2idx)

In [ ]:
import torch

embedding_dim = len(emb_pandas["embeddings"].iloc[0])

node_features = np.zeros((len(node2idx), embedding_dim), dtype=np.float32)

emb_matrix = np.vstack(emb_pandas["embeddings"].values).astype(np.float32)
positions = emb_pandas["internalId"].map(node2idx).values.astype(int)
node_features[positions] = emb_matrix

x = torch.tensor(node_features, dtype=torch.float)

print(f"x shape: {x.shape}")
print(f"Nodes without embedding: {(node_features == 0).all(axis=1).sum()}")

In [ ]:
def edges_to_tensors(df: pd.DataFrame, symmetric: bool = True) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Convert a remapped edge DataFrame to PyG-compatible tensors.
    Returns edge_index [2, E] and edge_attr [E].
    If symmetric=True, adds edges in both directions (A→B and B→A).
    """
    src = df["product_A"].values
    dst = df["product_B"].values
    weight = df["weight"].values

    if symmetric:
        # Stack both directions: A→B and B→A
        edge_index = np.stack([
            np.concatenate([src, dst]),
            np.concatenate([dst, src]),
        ])
        edge_attr = np.concatenate([weight, weight])
    else:
        edge_index = np.stack([src, dst])
        edge_attr = weight

    return (
        torch.tensor(edge_index, dtype=torch.long),
        torch.tensor(edge_attr, dtype=torch.float),
    )

# Convert remapped DataFrames to PyG tensors (symmetric for undirected graph)
train_edge_index, train_edge_attr = edges_to_tensors(train_remapped, symmetric=True)
val_edge_index, val_edge_attr = edges_to_tensors(val_remapped, symmetric=True)
test_edge_index, test_edge_attr = edges_to_tensors(test_remapped, symmetric=True)

num_nodes = len(node2idx)

print(f"Number of nodes: {num_nodes}")
print(f"Train edges (symmetric): {train_edge_index.shape[1]}")  
print(f"Val edges (symmetric): {val_edge_index.shape[1]}")
print(f"Test edges (symmetric): {test_edge_index.shape[1]}")
print(f"Train edge_index shape: {train_edge_index.shape}")     
print(f"Train edge_attr shape: {train_edge_attr.shape}") 

In [ ]:
def recommended_products_for_t2s_user(
    spark: SparkSession,
    customer_ids: list[str],
    publisher_ids: list[str],
    start_date: date,
    end_date: date,
) -> DataFrame:
    """Retrieve the relevant recommanded product for a t2s user"""

    df_merged_user = (
        spark.table("mirakl_data_platform.prod_data_platform_silver.t2s_merged_user")
        .filter(F.col("customerId").isin(customer_ids))
        .select(
            F.col("masterId").alias("userMasterId"),
            F.col("sourceId").alias("userId"),
        )
    )

    df_t2s_recommended_products = (
        spark.table("mirakl_data_platform.prod_data_platform_silver.ads_adlog_fct").alias("adlog")
        .filter(
            (F.col("adlog.__k8s_namespace") == "target2sell-prod")
            & F.col("adlog.publisherId").isin(publisher_ids)
            & F.col("adlog.__log_date").between(start_date, end_date)
            & (F.col("adlog.pageType") == "PRODUCT")
        )
        .join(
            spark.table("mirakl_data_platform.prod_data_platform_silver.t2s_tracking_product_display_page_event_fct").alias("t2s")
            .filter(
                F.col("t2s.customerId").isin(customer_ids)
                & F.col("t2s.emitted").between(start_date, end_date)
                & F.col("t2s.masterProductInternalId").isNotNull()
                & F.col("t2s.user.internalId").isNotNull()
            ),
            (F.col("adlog.productId") == F.col("t2s.fwProductId"))
            & (F.col("adlog.userCookie") == F.col("t2s.user.cookie"))
            & (F.col("adlog.customerId") == F.col("t2s.customerId"))
            & (F.col("t2s.emitted").between(
                    F.col("adlog.logInstant") - F.expr("INTERVAL 3 SECONDS"),
                    F.col("adlog.logInstant") + F.expr("INTERVAL 3 SECONDS"),
                )),
            how="inner",
        )
        .withColumn("userId", F.col("t2s.user.internalId"))
        .join(df_merged_user, on="userId", how="left")
        .withColumn("userId", F.coalesce(F.col("userMasterId"), F.col("userId")))
        .select(
            F.col("t2s.masterProductInternalId").alias("internalId"),
            F.col("t2s.emitted").alias("emitted"),
            F.col("userId"),
            F.col("t2s.customerId").alias("customerId"),
            F.col("adlog.executionId"),
            F.col("adlog.logInstant"),
            F.col("adlog.pageType"),
            F.col("adlog.searchTerm"),
            F.col("adlog.userOrganizeRank"),
            F.col("adlog.relevantProducts"),
            F.col("adlog.sponsoredProductPlacementExecutions"),
            F.col("adlog.productsReturned"),
        )
    )
    return df_t2s_recommended_products


In [ ]:
df_recommended_products_t2s_with_sessions_indexes = (
    df_split
    .join(
        recommended_products_for_t2s_user(spark, customer_ids, publisher_ids, start_date, end_date),
        on=["internalId", "emitted", "userId"],
        how="inner",
    )
)

df_recommended_products_t2s_with_sessions_indexes = df_recommended_products_t2s_with_sessions_indexes.cache()

In [ ]:
# For a first try we consider relevantProducts inside sponsoredProductPlacementExecutions

def positive_negative_edge_construction(df_recommended: DataFrame, df_split: DataFrame, df_train_edges: DataFrame, node2idx: dict[int, int], split: str) -> tuple:

    df_recommended_products_t2s_with_sessions = (
        df_recommended
        .filter(F.col("split") == split)
        .select(
            F.col("userId"),
            F.col("session_id"),
            F.col("internalId").alias("internalId_trigger"),
            F.col("executionId"),
            F.col("sponsoredProductPlacementExecutions"),
        )
        .join(
            df_split.filter(F.col("split") == split).select(
                F.col("userId"),
                F.col("session_id"),
                F.col("internalId").alias("internalId_session"),
            ),
            on=["userId", "session_id"],
            how="inner"
        )
    )

    df_recommended_exploded = (
        df_recommended_products_t2s_with_sessions
        .withColumn("placement", F.explode("sponsoredProductPlacementExecutions"))
        .withColumn(
            "top_relevant_products",
            F.slice(F.col("placement.relevantProducts"), 1, 12),
        )
        .select("*", F.posexplode("top_relevant_products").alias("rank_candidate", "candidate"))
        .select(
            F.col("userId"),
            F.col("session_id"),
            F.col("executionId"),
            F.col("internalId_trigger"),
            F.col("internalId_session"),
            F.col("candidate.internalId").alias("internalId_candidate"),
            (F.col("rank_candidate") + F.lit(1)).alias("rank_candidate"),  
        )
    ).cache()

    positive_candidates = (
        df_recommended_exploded
        .filter(F.col("internalId_session") == F.col("internalId_candidate"))
        .select(
            F.col("executionId"),
            F.col("internalId_trigger").alias("product_A"),
            F.col("internalId_candidate").alias("product_B"),
        )
        .distinct()
    ).cache()

    # Production MRR: static ranking-quality metric of the production algorithm itself.
    matched_rank = (
        df_recommended_exploded
        .filter(F.col("internalId_session") == F.col("internalId_candidate"))
        .groupBy("executionId")
        .agg(F.min("rank_candidate").alias("best_rank"))
        .withColumn("reciprocal_rank", F.lit(1.0) / F.col("best_rank"))
    )

    total_executions = df_recommended_exploded.select("executionId").distinct().count()
    
    # Single aggregation instead of two separate actions (.count() + .agg().collect()) on matched_rank,
    # which would otherwise recompute the filter+groupBy shuffle twice.
    match_stats = matched_rank.agg(
        F.count("*").alias("matched_executions"),
        F.sum("reciprocal_rank").alias("sum_reciprocal_rank"),
    ).collect()[0]
    matched_executions = match_stats["matched_executions"]
    sum_reciprocal_rank = match_stats["sum_reciprocal_rank"] or 0.0

    # Coverage-adjusted MRR: unmatched executions (no candidate found) contribute 0, denominator is ALL executions. (take all the samples into account)
    production_mrr_coverage = sum_reciprocal_rank / total_executions if total_executions > 0 else 0.0
    
    # Matched-only MRR: averages only over executions with a known positive, same convention as  (we don't take into account executions with only negatives)
    # (which only averages over groups with pos_count > 0 — see base_metrics.py::MRR_trigger._mrr).
    # For now, it is the only one directly comparable to the model's MRR_trigger metric.
    production_mrr_matched = sum_reciprocal_rank / matched_executions if matched_executions > 0 else 0.0
    coverage = matched_executions / total_executions if total_executions > 0 else 0.0

    print(
        f"[{split}] Production MRR (matched-only, comparable to MRR_trigger): {production_mrr_matched:.4f} | "
        f"Production MRR (coverage-adjusted): {production_mrr_coverage:.4f} | "
        f"{matched_executions}/{total_executions} executions matched (coverage={coverage:.2%})"
    )

    # All distinct candidate pairs (trigger, candidate)
    df_all_candidates = (
        df_recommended_exploded
        .select(
            F.col("executionId"),
            F.col("internalId_trigger").alias("product_A"),
            F.col("internalId_candidate").alias("product_B"),
            F.col("rank_candidate")
        )
        .groupBy("executionId", "product_A", "product_B")
        .agg(F.min("rank_candidate").alias("rank_candidate"))
    )

    df_positives_rank = (
        positive_candidates
        .join(
            df_all_candidates,
            on=["executionId", "product_A", "product_B"],
            how="inner"
        )
        .select("executionId", "product_A", "product_B", "rank_candidate")
    )

    # Existing edges in the graph as pairs (both directions)
    df_existing_pairs = (
        df_train_edges.select("product_A", "product_B")
        .union(
            df_train_edges.select(
                F.col("product_B").alias("product_A"),
                F.col("product_A").alias("product_B"),
            )
        )
        .distinct()
    )

    # Left join to distinguish true negatives (not in graph) from false negatives (already a graph edge)
    df_negative_labeled = (
        df_all_candidates
        .join(
            df_existing_pairs.withColumn("is_existing_edge", F.lit(True)),
            on=["product_A", "product_B"],
            how="left"
        )
        .fillna(False, subset=["is_existing_edge"])
    )

    # Cached: reused below for exec_only_negatives and the final left_semi sampling join.
    negative_candidates = (
        df_negative_labeled
        .filter(~F.col("is_existing_edge"))
        .select(
            F.col("executionId"),
            F.col("product_A"),
            F.col("product_B"),
            F.col("rank_candidate"),
        )
        .join(
            positive_candidates.select("executionId", "product_A", "product_B"),
            on=["executionId", "product_A", "product_B"],
            how="left_anti"
        )
    ).cache()

    deepest_pos_rank = (
        df_recommended_exploded
        .filter(F.col("internalId_session") == F.col("internalId_candidate"))
        .groupBy("executionId")
        .agg(F.max("rank_candidate").alias("deepest_pos_rank"))
    )

    # Reduce negative volume: keep all executions with positives + 10% of pure-negative executions
    exec_with_positives = positive_candidates.select("executionId").distinct()

    negative_candidates_matched = (
        negative_candidates
        .join(F.broadcast(deepest_pos_rank), on="executionId", how="inner")
        .filter(F.col("rank_candidate") <= F.col("deepest_pos_rank"))
        .select("executionId", "product_A", "product_B")
    )

    exec_only_negatives = (
        negative_candidates.select("executionId").distinct()
        .join(exec_with_positives, on="executionId", how="left_anti")
        .sample(fraction=0.1, seed=42)
    )

    negative_candidates_pure_negative = (
        negative_candidates
        .join(F.broadcast(exec_only_negatives), on="executionId", how="left_semi")
        .select("executionId", "product_A", "product_B")
    )

    negative_candidates_sampled = negative_candidates_matched.union(negative_candidates_pure_negative)
    
    
    # Convert to pandas separately — no cross-join, sampling handled in dataloader
    positive_pandas = positive_candidates.toPandas()
    negative_pandas = negative_candidates_sampled.toPandas()

    # Encode executionId consistently across both tables
    all_exec_ids = pd.concat([
        positive_pandas["executionId"],
        negative_pandas["executionId"]
    ]).unique()
    exec2code = {eid: i for i, eid in enumerate(all_exec_ids)}
    # all_exec_ids is already ordered by code (index i <-> code i), so it IS code2exec.
    code2exec = all_exec_ids.tolist()

    positive_pandas["exec_code"] = positive_pandas["executionId"].map(exec2code)
    negative_pandas["exec_code"] = negative_pandas["executionId"].map(exec2code)

    # Remap product IDs to contiguous indices
    for col in ["product_A", "product_B"]:
        positive_pandas[col] = positive_pandas[col].map(node2idx)
        negative_pandas[col] = negative_pandas[col].map(node2idx)

    positive_pandas = positive_pandas.dropna(subset=["product_A", "product_B", "exec_code"])
    negative_pandas = negative_pandas.dropna(subset=["product_A", "product_B", "exec_code"])

    # 3D tensors [3, N] : [exec_code, trigger, product]
    pos_edge_index = torch.tensor(
        positive_pandas[["exec_code", "product_A", "product_B"]].values.T,
        dtype=torch.long
    )

    neg_edge_index = torch.tensor(
        negative_pandas[["exec_code", "product_A", "product_B"]].values.T,
        dtype=torch.long
    )

    print(f"pos_edge_index shape: {pos_edge_index.shape}")
    print(f"neg_edge_index shape: {neg_edge_index.shape}")

    return (
        pos_edge_index,
        neg_edge_index,
        production_mrr_matched,
        production_mrr_coverage,
        exec2code,
        code2exec,
        df_positives_rank,
        negative_candidates,
        df_all_candidates,
    )


train_pos_edge_index, train_neg_edge_index, train_production_mrr, train_production_mrr_coverage, train_exec2code, train_code2exec, _, _, _ = positive_negative_edge_construction(df_recommended_products_t2s_with_sessions_indexes, df_split, df_train_edges, node2idx, "train")
val_pos_edge_index, val_neg_edge_index, val_production_mrr, val_production_mrr_coverage, val_exec2code, val_code2exec, val_positives_rank, val_negative_candidates, val_all_candidates = positive_negative_edge_construction(df_recommended_products_t2s_with_sessions_indexes, df_split, df_train_edges, node2idx, "val")
test_pos_edge_index, test_neg_edge_index, test_production_mrr, test_production_mrr_coverage, test_exec2code, test_code2exec, _, _, _ = positive_negative_edge_construction(df_recommended_products_t2s_with_sessions_indexes, df_split, df_train_edges, node2idx, "test")

print(f"Train pos_edge_index shape: {train_pos_edge_index.shape}")
print(f"Train neg_edge_index shape: {train_neg_edge_index.shape}")

print(f"Val pos_edge_index shape: {val_pos_edge_index.shape}")
print(f"Val neg_edge_index shape: {val_neg_edge_index.shape}")

print(f"Test pos_edge_index shape: {test_pos_edge_index.shape}")
print(f"Test neg_edge_index shape: {test_neg_edge_index.shape}")

print(f"Train production MRR (matched-only, comparable to MRR_trigger): {train_production_mrr:.4f} | coverage-adjusted: {train_production_mrr_coverage:.4f}")
print(f"Val production MRR (matched-only, comparable to MRR_trigger): {val_production_mrr:.4f} | coverage-adjusted: {val_production_mrr_coverage:.4f}")
print(f"Test production MRR (matched-only, comparable to MRR_trigger): {test_production_mrr:.4f} | coverage-adjusted: {test_production_mrr_coverage:.4f}")

Creation of sessions_raw_val -> sessions with all the products

In [ ]:
val_triggers = (
    df_recommended_products_t2s_with_sessions_indexes
    .filter(F.col("split") == "val")
    .select("userId", "session_id", "executionId", F.col("internalId").alias("trigger_internal_id"))
    .distinct()
)

In [ ]:
val_session_products = (
    val_triggers
    .join(
        df_split.filter(F.col("split") == "val").select("userId", "session_id", "internalId", "emitted"),
        on=["userId", "session_id"],
        how="inner",
    )
    .groupBy("userId", "session_id", "executionId", "trigger_internal_id")
    .agg(
        F.collect_list(
            F.struct(F.col("internalId").alias("internal_id"), F.col("emitted"))
        ).alias("session_products")
    )
)

val_exec2code_pandas = pd.DataFrame(val_exec2code.items(), columns=["executionId", "exec_code"])
df_val_exec2code = spark.createDataFrame(val_exec2code_pandas)

sessions_raw_val = (
    val_session_products
    .join(df_val_exec2code, on="executionId", how="inner")
    .select(
        "exec_code",
        "trigger_internal_id",
        F.col("executionId").alias("execution_id"),
        "session_id",
        "session_products",
    )
)

print(f"sessions_raw_val: {sessions_raw_val.count()} triggers")
sessions_raw_val.printSchema()

display(sessions_raw_val.limit(1))

In [ ]:
sessions_raw_val.write.mode("overwrite").parquet(
    "s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/sessions_raw_val.parquet"
)
print("sessions_raw_val.parquet uploaded")

Creation of prod_results_val -> what productions returns with positives and negatives

In [ ]:
val_positive_prod_products = (
    val_triggers
    .withColumnRenamed("trigger_internal_id", "product_A")
    .join(
        val_positives_rank,
        on=["executionId", "product_A"],
        how="left"
    )
    .filter(F.col("product_B").isNotNull())
    .groupBy("executionId", "product_A")
    .agg(
        F.collect_list(
            F.struct(F.col("product_B").alias("internal_id"), F.col("rank_candidate").alias("rank"))
        ).alias("positives")
    )
)

val_negatives_prod_products = (
    val_triggers
    .withColumnRenamed("trigger_internal_id", "product_A")
    .join(
        val_negative_candidates,
        on=["executionId", "product_A"],
        how="left"
    )
    .filter(F.col("product_B").isNotNull())
    .groupBy("executionId", "product_A")
    .agg(
        F.collect_list(
            F.struct(F.col("product_B").alias("internal_id"), F.col("rank_candidate").alias("rank"))
        ).alias("negatives")
    )
)

val_all_prod_products = (
    val_triggers
    .withColumnRenamed("trigger_internal_id", "product_A")
    .join(
        val_all_candidates,
        on=["executionId", "product_A"],
        how="left"
    )
    .filter(F.col("product_B").isNotNull())
    .groupBy("executionId", "product_A")
    .agg(
        F.collect_list(
            F.struct(F.col("product_B").alias("internal_id"), F.col("rank_candidate").alias("rank"))
        ).alias("products_returned")
    )
)

prod_results_val = (
    val_triggers
    .withColumnRenamed("trigger_internal_id", "product_A")
    .join(
        val_positive_prod_products,
        on=["executionId", "product_A"],
        how="left"
    )
    .join(
        val_negatives_prod_products,
        on=["executionId", "product_A"],
        how="left"
    )
    .join(
        val_all_prod_products,
        on=["executionId", "product_A"],
        how="left"
    )
    .join(df_val_exec2code, on="executionId", how="inner")
    .select(
        "exec_code",
        F.col("product_A").alias("trigger_internal_id"),
        F.col("executionId").alias("execution_id"),
        F.coalesce(F.col("positives"), F.array()).alias("positives"),
        F.coalesce(F.col("negatives"), F.array()).alias("negatives"),
        F.coalesce(F.col("products_returned"), F.array()).alias("products_returned"),
    )
)

print(f"prod_results_val: {prod_results_val.count()} triggers")
prod_results_val.printSchema()

display(prod_results_val.limit(1))

In [ ]:
prod_results_val.write.mode("overwrite").parquet(
    "s3://mirakl-data-science-tmp2/nbraun/datasets/coview-mdm/prod_results_val.parquet"
)
print("prod_results_val.parquet uploaded")

In [ ]:
from torch_geometric.data import Data


def build_pyg_data(
    num_nodes: int,
    train_edge_index: torch.Tensor,
    train_neg_edge_index: torch.Tensor,
    train_pos_edge_index: torch.Tensor,
    train_edge_attr: torch.Tensor,
    val_pos_edge_index: torch.Tensor,
    val_neg_edge_index: torch.Tensor,
    test_pos_edge_index: torch.Tensor,
    test_neg_edge_index: torch.Tensor,
    x: torch.Tensor = None,
) -> Data:
    """
    Assemble all tensors into a single PyG Data object.
    """
    data = Data(
        edge_index=train_edge_index,
        edge_attr=train_edge_attr,
        num_nodes=num_nodes,
    )

    if x is not None:
        data.x = x

    data.train_neg_edge_index = train_neg_edge_index
    data.train_pos_edge_index = train_pos_edge_index

    # Evaluation edges
    data.val_pos_edge_index = val_pos_edge_index
    data.val_neg_edge_index = val_neg_edge_index
    data.test_pos_edge_index = test_pos_edge_index
    data.test_neg_edge_index = test_neg_edge_index

    return data

In [ ]:
num_nodes = len(node2idx)

data = build_pyg_data(
    num_nodes=num_nodes,
    train_edge_index=train_edge_index,
    train_neg_edge_index = train_neg_edge_index,
    train_pos_edge_index = train_pos_edge_index,
    train_edge_attr=train_edge_attr,
    val_pos_edge_index=val_pos_edge_index,
    val_neg_edge_index=val_neg_edge_index,
    test_pos_edge_index=test_pos_edge_index,
    test_neg_edge_index=test_neg_edge_index,
    x=x,
)

print(data)

In [ ]:
import os
import json
from torch_geometric.data import InMemoryDataset


class CoViewDataset(InMemoryDataset):
    """
    PyG InMemoryDataset for the co-view graph.
    Saves the Data object + node mapping to disk for fast reloading.
    """
    def __init__(self, root: str, data_obj: Data = None, node2idx: dict = None, transform=None, force_reload: bool = False):
        self._data_obj = data_obj
        self._node2idx = node2idx
        super().__init__(root, transform, force_reload=force_reload)
        self.load(self.processed_paths[0])

    @property
    def processed_file_names(self):
        return ["data.pt"]

    def process(self):
        self.save([self._data_obj], self.processed_paths[0])

        if self._node2idx is not None:
            mapping_path = os.path.join(self.processed_dir, "node2idx.json")
            with open(mapping_path, "w") as f:
                json.dump({int(k): int(v) for k, v in self._node2idx.items()}, f)

In [ ]:
dataset = CoViewDataset(
    root="/dbfs/tmp/nbraun/datasets/coview-mdm-v2",
    data_obj=data,
    node2idx=node2idx,
    force_reload=True,
)

print(f"Dataset saved to /dbfs/tmp/nbraun/datasets/coview-mdm-v2/")
print(dataset[0])

In [ ]:
import boto3

s3 = boto3.client("s3")

s3.upload_file(
    "/dbfs/tmp/nbraun/datasets/coview-mdm-v2/processed/data.pt",
    "mirakl-data-science-tmp2",
    "nbraun/datasets/coview-mdm/data.pt"
)
print("data.pt uploaded")

s3.upload_file(
    "/dbfs/tmp/nbraun/datasets/coview-mdm-v2/processed/node2idx.json",
    "mirakl-data-science-tmp2",
    "nbraun/datasets/coview-mdm/node2idx.json"
)
print("node2idx.json uploaded")

In [ ]:
exec_mappings = {
    "train": {
        "exec2code": {str(eid): code for eid, code in train_exec2code.items()},
        "code2exec": train_code2exec,
    },
    "val": {
        "exec2code": {str(eid): code for eid, code in val_exec2code.items()},
        "code2exec": val_code2exec,
    },
    "test": {
        "exec2code": {str(eid): code for eid, code in test_exec2code.items()},
        "code2exec": test_code2exec,
    },
}

exec_mappings_path = "/dbfs/tmp/nbraun/datasets/coview-mdm-v2/processed/exec_mappings.json"
with open(exec_mappings_path, "w") as f:
    json.dump(exec_mappings, f)

s3.upload_file(
    exec_mappings_path,
    "mirakl-data-science-tmp2",
    "nbraun/datasets/coview-mdm/exec_mappings.json"
)
print("exec_mappings.json uploaded")